# Congratulations, You Have a DAG. Now What?

**Monetary Policy Surprises and Equity Returns** (Bernanke & Kuttner 2005)

We take a well-known causal structure from the monetary-economics literature, instantiate it with three competing functional forms (linear, regime-switching, nonlinear), and use PyMC to recover the correct specification via LOO model comparison. A toy trading strategy shows that the right DAG with the wrong functional form loses money exactly when it matters.

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

plt.rcParams.update({"figure.figsize": (10, 5), "font.size": 12})
az.style.use("arviz-darkgrid")
%config InlineBackend.figure_format = "retina"

## The DAG: Bernanke-Kuttner Channels

Two channels transmit monetary policy surprises to equity returns:

```
  Macro news (MN) ──────────────────────────┐
       │                                     │
       ▼                                     ▼
  MP_surprise ──► Discount rates (DR) ──► Equity return (R_eq)
       │                                     ▲
       └──────► Expected cash flows (ECF) ───┘
                                             ▲
  Risk sentiment (RS) ───────────────────────┘
       └──────────────► DR
```

| Variable | Symbol | Interpretation |
|----------|--------|----------------|
| Monetary policy surprise | MP_surp | Futures-implied surprise in target rate |
| Discount rates | DR | Change in risk-free discounting / yields |
| Expected cash flows | ECF | Cash-flow news / expected earnings |
| Risk sentiment | RS | Time-varying risk appetite |
| Macro news | MN | Information moving both policy and cash flows |
| Equity returns | R_eq | Equity index return |

The structural model:

$$DR = f_1(\text{MP\_surp}, RS) + \varepsilon_1$$
$$ECF = f_2(\text{MP\_surp}, MN) + \varepsilon_2$$
$$R_{eq} = f_3(DR, ECF, RS) + \varepsilon_3$$

The DAG is the same across all three specifications. The question is: **what are $f_1, f_2, f_3$?**

## Data Generating Processes

Three functional forms instantiate the same DAG:

In [ ]:
def dgp_mp_linear(n=2000, seed=42):
    """Form A: Linear (baseline)."""
    rng = np.random.default_rng(seed)
    RS = rng.normal(0, 1, n)
    MN = rng.normal(0, 1, n)
    MP_surp = 0.3 * MN + rng.normal(0, 1, n)
    DR = 0.8 * MP_surp + 0.6 * RS + rng.normal(0, 1, n)
    ECF = -0.4 * MP_surp + 0.7 * MN + rng.normal(0, 1, n)
    R_eq = -0.6 * DR + 0.8 * ECF + 0.5 * RS + rng.normal(0, 1, n)
    return {"RS": RS, "MN": MN, "MP_surp": MP_surp, "DR": DR, "ECF": ECF, "R_eq": R_eq}


def dgp_mp_regime(n=2000, seed=42):
    """Form B: Regime-switching (tightening vs easing asymmetry)."""
    rng = np.random.default_rng(seed)
    RS = rng.normal(0, 1, n)
    MN = rng.normal(0, 1, n)
    MP_surp = 0.3 * MN + rng.normal(0, 1, n)
    tightening = MP_surp > 0
    DR = np.where(tightening, 1.2 * MP_surp, 0.5 * MP_surp) + 0.6 * RS + rng.normal(0, 1, n)
    ECF = np.where(tightening, -0.3 * MP_surp, -0.7 * MP_surp) + 0.7 * MN + rng.normal(0, 1, n)
    stress = RS > 1.0
    R_eq = (
        -0.6 * DR + 0.8 * ECF + 0.5 * RS
        + np.where(stress, -1.0 * np.abs(MP_surp), 0.0)
        + rng.normal(0, 1, n)
    )
    return {"RS": RS, "MN": MN, "MP_surp": MP_surp, "DR": DR, "ECF": ECF, "R_eq": R_eq}


def dgp_mp_nonlinear(n=2000, seed=42):
    """Form C: Nonlinear interaction (tanh saturation + risk premia interaction)."""
    rng = np.random.default_rng(seed)
    RS = rng.normal(0, 1, n)
    MN = rng.normal(0, 1, n)
    MP_surp = 0.3 * MN + rng.normal(0, 1, n)
    DR = (
        1.0 * np.tanh(0.8 * MP_surp)
        + 0.7 * np.tanh(0.6 * RS)
        + 0.4 * MP_surp * RS
        + rng.normal(0, 1, n)
    )
    ECF = -0.6 * np.tanh(0.7 * MP_surp) + 0.8 * np.tanh(0.6 * MN) + rng.normal(0, 1, n)
    R_eq = -0.7 * DR + 0.9 * ECF + 0.6 * RS + rng.normal(0, 1, n)
    return {"RS": RS, "MN": MN, "MP_surp": MP_surp, "DR": DR, "ECF": ECF, "R_eq": R_eq}

In [ ]:
N = 2000
data_true = dgp_mp_regime(n=N, seed=RANDOM_SEED)
df = pd.DataFrame(data_true)
df.describe().round(2)

In [ ]:
tightening = df["MP_surp"].values > 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, mask, color in [("Tightening", tightening, "C3"), ("Easing", ~tightening, "C0")]:
    axes[0].scatter(df["MP_surp"][mask], df["DR"][mask], alpha=0.2, s=8, label=label, color=color)
    axes[1].scatter(df["MP_surp"][mask], df["ECF"][mask], alpha=0.2, s=8, label=label, color=color)

axes[0].set(xlabel="MP_surp", ylabel="DR", title="Discount-Rate Channel")
axes[1].set(xlabel="MP_surp", ylabel="ECF", title="Cash-Flow Channel")
for ax in axes:
    ax.legend(fontsize=9)
fig.suptitle("Asymmetric slopes visible in the raw data", fontsize=13)
plt.tight_layout()

## PyMC Models

All three models encode the same DAG. They differ only in the functional forms of $f_1, f_2, f_3$.

In [ ]:
# --- Model A: Linear ---
mp_vals = df["MP_surp"].values
rs_vals = df["RS"].values
mn_vals = df["MN"].values
dr_vals = df["DR"].values
ecf_vals = df["ECF"].values
req_vals = df["R_eq"].values

with pm.Model() as model_linear:
    # DR = a1*MP + a2*RS + noise
    a1 = pm.Normal("a1", mu=0, sigma=2)
    a2 = pm.Normal("a2", mu=0, sigma=2)
    sigma_dr = pm.HalfNormal("sigma_dr", sigma=2)
    pm.Normal("DR", mu=a1 * mp_vals + a2 * rs_vals, sigma=sigma_dr, observed=dr_vals)

    # ECF = b1*MP + b2*MN + noise
    b1 = pm.Normal("b1", mu=0, sigma=2)
    b2 = pm.Normal("b2", mu=0, sigma=2)
    sigma_ecf = pm.HalfNormal("sigma_ecf", sigma=2)
    pm.Normal("ECF", mu=b1 * mp_vals + b2 * mn_vals, sigma=sigma_ecf, observed=ecf_vals)

    # R_eq = c1*DR + c2*ECF + c3*RS + noise
    c1 = pm.Normal("c1", mu=0, sigma=2)
    c2 = pm.Normal("c2", mu=0, sigma=2)
    c3 = pm.Normal("c3", mu=0, sigma=2)
    sigma_req = pm.HalfNormal("sigma_req", sigma=2)
    pm.Normal("R_eq", mu=c1 * dr_vals + c2 * ecf_vals + c3 * rs_vals, sigma=sigma_req, observed=req_vals)

In [ ]:
# --- Model B: Regime-switching ---
tight = (mp_vals > 0).astype(float)
stress = (rs_vals > 1.0).astype(float)

with pm.Model() as model_regime:
    # DR: regime-dependent slope on MP
    a1_t = pm.Normal("a1_tight", mu=0, sigma=2)
    a1_e = pm.Normal("a1_ease", mu=0, sigma=2)
    a2 = pm.Normal("a2", mu=0, sigma=2)
    sigma_dr = pm.HalfNormal("sigma_dr", sigma=2)
    dr_mu = (a1_t * tight + a1_e * (1 - tight)) * mp_vals + a2 * rs_vals
    pm.Normal("DR", mu=dr_mu, sigma=sigma_dr, observed=dr_vals)

    # ECF: regime-dependent slope on MP
    b1_t = pm.Normal("b1_tight", mu=0, sigma=2)
    b1_e = pm.Normal("b1_ease", mu=0, sigma=2)
    b2 = pm.Normal("b2", mu=0, sigma=2)
    sigma_ecf = pm.HalfNormal("sigma_ecf", sigma=2)
    ecf_mu = (b1_t * tight + b1_e * (1 - tight)) * mp_vals + b2 * mn_vals
    pm.Normal("ECF", mu=ecf_mu, sigma=sigma_ecf, observed=ecf_vals)

    # R_eq: base channels + stress amplification
    c1 = pm.Normal("c1", mu=0, sigma=2)
    c2 = pm.Normal("c2", mu=0, sigma=2)
    c3 = pm.Normal("c3", mu=0, sigma=2)
    c_stress = pm.Normal("c_stress", mu=0, sigma=2)
    sigma_req = pm.HalfNormal("sigma_req", sigma=2)
    req_mu = c1 * dr_vals + c2 * ecf_vals + c3 * rs_vals + c_stress * stress * np.abs(mp_vals)
    pm.Normal("R_eq", mu=req_mu, sigma=sigma_req, observed=req_vals)

In [ ]:
# --- Model C: Nonlinear (tanh saturation + interaction) ---
with pm.Model() as model_nonlinear:
    # DR: tanh saturation + MP*RS interaction
    a1 = pm.Normal("a1", mu=0, sigma=2)
    a2 = pm.Normal("a2", mu=0, sigma=2)
    a3 = pm.Normal("a3", mu=0, sigma=2)
    a_s1 = pm.HalfNormal("a_scale1", sigma=2)
    a_s2 = pm.HalfNormal("a_scale2", sigma=2)
    sigma_dr = pm.HalfNormal("sigma_dr", sigma=2)
    dr_mu = a1 * pm.math.tanh(a_s1 * mp_vals) + a2 * pm.math.tanh(a_s2 * rs_vals) + a3 * mp_vals * rs_vals
    pm.Normal("DR", mu=dr_mu, sigma=sigma_dr, observed=dr_vals)

    # ECF: tanh saturation
    b1 = pm.Normal("b1", mu=0, sigma=2)
    b2 = pm.Normal("b2", mu=0, sigma=2)
    b_s1 = pm.HalfNormal("b_scale1", sigma=2)
    b_s2 = pm.HalfNormal("b_scale2", sigma=2)
    sigma_ecf = pm.HalfNormal("sigma_ecf", sigma=2)
    ecf_mu = b1 * pm.math.tanh(b_s1 * mp_vals) + b2 * pm.math.tanh(b_s2 * mn_vals)
    pm.Normal("ECF", mu=ecf_mu, sigma=sigma_ecf, observed=ecf_vals)

    # R_eq: linear combination
    c1 = pm.Normal("c1", mu=0, sigma=2)
    c2 = pm.Normal("c2", mu=0, sigma=2)
    c3 = pm.Normal("c3", mu=0, sigma=2)
    sigma_req = pm.HalfNormal("sigma_req", sigma=2)
    pm.Normal("R_eq", mu=c1 * dr_vals + c2 * ecf_vals + c3 * rs_vals, sigma=sigma_req, observed=req_vals)

## Sampling

In [ ]:
models = {"linear": model_linear, "regime": model_regime, "nonlinear": model_nonlinear}
traces = {}

for name, model in models.items():
    print(f"Sampling {name}...")
    with model:
        traces[name] = pm.sample(
            draws=2000, tune=1000, chains=4,
            random_seed=RANDOM_SEED, target_accept=0.95,
        )
        pm.compute_log_likelihood(traces[name])

## Model Comparison (LOO)

In [ ]:
comparison = az.compare(traces, ic="loo")
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
az.plot_compare(comparison, ax=ax, insample_dev=False)
ax.set_title("LOO Model Comparison: Same DAG, Three Functional Forms")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for i, (name, model) in enumerate(models.items()):
    with model:
        ppc = pm.sample_posterior_predictive(traces[name], random_seed=RANDOM_SEED)
    az.plot_ppc(ppc, var_names=["R_eq"], ax=axes[i], num_pp_samples=200, kind="cumulative")
    axes[i].set_title(f"{name}")
fig.suptitle("Posterior Predictive Checks: R_eq", fontsize=13)
plt.tight_layout()

## Toy Trading Strategy

On each policy-event day: predict $R_{eq}$ from the model's posterior mean. Go long if predicted > 0, short if < 0. Position size proportional to |prediction|, capped at 1.

In [ ]:
def compute_strategy_pnl(trace, model, actual_req):
    with model:
        ppc = pm.sample_posterior_predictive(trace, random_seed=RANDOM_SEED)
    pred = ppc.posterior_predictive["R_eq"].mean(dim=["chain", "draw"]).values
    signal = np.sign(pred)
    size = np.clip(np.abs(pred) / np.std(pred), 0, 1)
    pnl = signal * size * actual_req
    return np.cumsum(pnl), pred

results = {}
for name, model in models.items():
    results[name] = compute_strategy_pnl(traces[name], model, req_vals)

In [ ]:
stress_mask = rs_vals > 1.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (cum_pnl, _) in results.items():
    axes[0].plot(cum_pnl, label=name, alpha=0.9)
axes[0].set(xlabel="Event index", ylabel="Cumulative P&L", title="Cumulative P&L by Model")
axes[0].legend()
axes[0].axhline(0, color="black", ls="--", alpha=0.3)

colors = ["C0", "C1", "C2"]
for i, (name, (_, pred)) in enumerate(results.items()):
    signal = np.sign(pred)
    size = np.clip(np.abs(pred) / np.std(pred), 0, 1)
    pnl_stress = (signal * size * req_vals)[stress_mask].sum()
    axes[1].bar(name, pnl_stress, color=colors[i], alpha=0.8)
axes[1].set(title="Total P&L During Stress (RS > 1)", ylabel="Total P&L")

plt.tight_layout()

## Takeaways

1. Same DAG (Bernanke-Kuttner channels), three functional forms, dramatically different predictions.
2. LOO correctly identifies the regime-switching form as the best specification.
3. The linear model's residuals are well-behaved on average but catastrophically wrong in stress.
4. Wrong functional form + right DAG = money left on the table exactly when it matters.
5. **The pipeline**: DAG (from theory or discovery) $\to$ competing PyMC models $\to$ LOO selection $\to$ decision.